# GRU Deep-Learning Drift Detection and Adaptation

This notebook follows the uploaded drift-adaptation experiment's chronological streaming design and expands it to five GRU-only strategies and three detectors: KS, PSI, and Page-Hinkley.

**Important:** the included dataset is synthetic, with bounded sensor readings and autocorrelation. It is a controlled test benchmark, not real industrial telemetry. Ground-truth drift annotations are reserved for offline evaluation.

## Drift detectors
- **KS test:** statistical test for shifts in sensor feature distributions.
- **PSI:** magnitude of distribution shift relative to a reference window.
- **Page-Hinkley:** sequential detector on normalized baseline-GRU prediction errors, useful for relational/concept drift.

The detectors use observed sensors and predictions/errors only. Ground-truth drift labels are reserved for offline evaluation.

## 1. Setup

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# If opened from the notebooks/ folder, move to repository root.
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / "data" / "industrial_sensor_drift_dataset.csv"
RESULTS_PATH = ROOT / "results"
print("Repository root:", ROOT)
print("Dataset:", DATA_PATH)

## 2. Load and validate the sensor dataset

In [ ]:
if not DATA_PATH.exists():
    from src.data_generator import generate_dataset
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    generate_dataset().to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
features = ["temperature_C", "pressure_bar", "vibration_mm_s_rms",
            "load_pct", "humidity_pct", "motor_speed_rpm", "operating_hours"]
target = "health_deterioration"
assert set(features + [target]).issubset(df.columns)
print("Rows:", len(df), "| columns:", len(df.columns))
display(df.head())
display(df[features + [target]].describe().T.round(3))

## 3. Inspect the designed regimes (offline ground truth)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df["health_deterioration"], linewidth=.8)
axes[0].set_ylabel("Deterioration (0–100)")
axes[0].set_title("Synthetic target over time")
for boundary, label in [(24000,"Data drift"),(30000,"Concept drift"),(42000,"Combined drift")]:
    axes[0].axvline(boundary, linestyle="--", alpha=.8)
    axes[0].text(boundary+25, axes[0].get_ylim()[1]*.92, label, rotation=90, va="top")
for col in features:
    axes[1].plot(df.index, (df[col]-df[col].mean())/df[col].std(), label=col, linewidth=.7)
axes[1].set_ylabel("Standardized sensor")
axes[1].set_xlabel("Row / hour")
axes[1].legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 4. Experiment protocol

- Initial GRU uses rows 0–14,399 for training and 14,400–15,999 for validation/reference.
- Streaming starts at row 3,200.
- Each batch is predicted **before** its labels are used.
- A Page-Hinkley detector on the static baseline's prediction error provides a shared alarm schedule.
- The drift truth columns are not passed to the live detector or adaptation functions.
- Adaptation begins only after the detector has accumulated the configured minimum labelled observations.

## 5. Run five GRU strategies

In [ ]:
from src.experiment import run_experiment

metrics, events = run_experiment(
    data_path=str(DATA_PATH),
    out_dir=str(RESULTS_PATH),
    fast=True,       # quick run; use --full from a terminal for longer training
    seed=42
)
display(metrics)
display(events)

## 6. Compare strategies by regime

In [ ]:
overall = metrics[metrics["Phase"] == "overall"].sort_values("MAE")
display(overall)
pivot = metrics.pivot(index="Strategy", columns="Phase", values="MAE")
display(pivot.round(3))
ax = overall.set_index("Strategy")[["MAE","RMSE"]].plot(kind="bar", figsize=(12,4))
ax.set_title("Overall prediction error (lower is better)")
ax.set_ylabel("Error in deterioration-score points")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 7. Plot actual vs model predictions and event markers

In [ ]:
pred_path = RESULTS_PATH / "predictions.csv"
pred = pd.read_csv(pred_path, parse_dates=["timestamp"])
truth = pd.read_csv(RESULTS_PATH / "ground_truth_offline_only.csv", parse_dates=["timestamp"])
view_lo, view_hi = 15000, min(60000, len(pred))
fig, ax = plt.subplots(figsize=(15,5))
ax.plot(pred["row"][view_lo:view_hi], pred["actual"][view_lo:view_hi], color="black", linewidth=1.5, label="Actual")
for col in [c for c in pred.columns if c not in {"row","timestamp","actual"}]:
    ax.plot(pred["row"][view_lo:view_hi], pred[col][view_lo:view_hi], linewidth=.9, alpha=.8, label=col)
for point in [24000,30000,42000]:
    ax.axvline(point, linestyle="--", color="crimson", alpha=.7)
ax.set_title("Test-then-train predictions; vertical lines are offline known drift starts")
ax.set_xlabel("Row")
ax.set_ylabel("Deterioration score (0–100)")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 8. Scientific interpretation checklist

1. Do not describe synthetic data as real plant data.
2. Compare MAE/RMSE/R² for the new regime **and** old-regime retention; replay can preserve old patterns while slowing adaptation.
3. Run several random seeds and report mean ± standard deviation before drawing conclusions.
4. The synthetic target is generated from the sensor variables; this is a designed proxy target, not a measured probability of failure.
5. For stronger external validity, repeat the experiment on a public dataset such as NASA C-MAPSS / IMS Bearings or another domain-matched industrial benchmark, following its licence and official split protocol.